In [1]:
#from google.colab import drive
%load_ext autoreload
%autoreload 2
import sys
import os

# Mount Google Drive
#drive.mount('/content/drive')

# Add your project folder to Python's search path
#project_path = '/content/drive/My Drive/Colab Notebooks/'
#sys.path.append(project_path)
#os.chdir(project_path)

In [2]:
import numpy as np
from lib import data_prep as dp
from lib import metrics
from lib import model 
from lib import plotting
import pandas as pd
from scipy.stats import norm
from sklearn.gaussian_process.kernels import Matern,RBF, ConstantKernel as C
import torch
from sklearn.preprocessing import StandardScaler,MinMaxScaler, PowerTransformer
import matplotlib.pyplot as plt
import math
from sklearn.pipeline import make_pipeline
from sklearn.decomposition import KernelPCA


In [3]:
scaler = StandardScaler()

all_y = []

In [5]:
input_registry, output_registry = dp.load_new_xy("../data/weekly_data/inputs.txt", "../data/weekly_data/outputs.txt")



In [6]:
def pca(q, x_raw, y_raw):
    kpca_linear = KernelPCA(n_components=1, kernel="linear")
    X_linear = kpca_linear.fit_transform(x_raw)
    plt.scatter(X_linear, y_scaled)
    plt.xlabel("X")
    plt.ylabel("Y")
    plt.title(f"Target Distribution Along the First Principal Component (PC1 vs. y) - Q{q}") 
    plt.show()
    X_centered = x_raw - np.mean(x_raw, axis=0)
    # 2. Covariance matrix
    cov_matrix = np.cov(X_centered, rowvar=False)
    
    # 3. Eigen-decomposition
    eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)    
    kpca_eigenvalues = kpca_linear.eigenvalues_
    explained_variance_ratio = eigenvalues[0] / np.sum(eigenvalues)  
    print(f"Explained Variance Ratio of PC1: {explained_variance_ratio * 100:.2f}%")

In [7]:
# =====================================================================
# #1. Loading Data
# =====================================================================
q = 1
noise_assumption = 1e-6  # Ensure globally tracked parameters are defined upfront
normalize_data = True
minMaxScaler = MinMaxScaler(feature_range=(-1, 1))
# Load initial baseline files explicitly
initial_x_raw, initial_y_raw = dp.load_xy(
    f"../data/initial_data/function_{q}/initial_inputs.npy", 
    f"../data/initial_data/function_{q}/initial_outputs.npy"
)

# Extract newly recorded registry submissions for this question
new_x = np.array(input_registry[f"Q{q}"])
new_y = np.array(output_registry[f"Q{q}"])

# Combine baseline and new batches into a single active matrix
x_raw = np.vstack([initial_x_raw, new_x])
y_raw = np.append(initial_y_raw, new_y)


# FIXED: Generate grid *after* x_raw has been initialized to avoid NameError
x_grid = dp.generate_x_grid(x_raw.shape[1], m=1000)

# Track pipeline iteration metrics
all_y.append({
    "max": initial_y_raw.max(), 
    "st": initial_y_raw.std(), 
    "mean": y_raw.mean(), 
    "X_shape": x_raw.shape
})

# =====================================================================
# #2. Data Preparation
# =====================================================================
# Robust Outlier elimination using your new Median Absolute Deviation filter
#x_raw, y_raw = dp.remove_outliers(x_raw, y_raw, method="mad", THRESHOLD=3)
plotting.plot_xy(x_raw, y_raw)


y_scaler_pipeline = make_pipeline(
    PowerTransformer(method='yeo-johnson'),
    #minMaxScaler
)
#y_scaled = y_scaler_pipeline.fit_transform(y_raw.reshape(-1, 1)).flatten()


x_new = np.sqrt(x_raw[:, 0]**2 + x_raw[:, 1]**2).reshape(-1,1)
x_minMaxScaler = MinMaxScaler(feature_range=(0, 1))
x_new = x_minMaxScaler.fit_transform(x_new)
#x_raw = np.hstack((x_raw, x_new))

x_grid_new = np.sqrt(x_grid[:, 0]**2 + x_grid[:, 1]**2)
#x_grid = np.column_stack((x_grid, x_grid_new))


#y_raw = y_raw + abs(y_raw.min()) 
y_transformed = np.log10(np.clip(y_raw , 1e-160,1))
#y_transformed = dp.transform_targets(y_raw)
minMaxScaler = MinMaxScaler(feature_range=(0, 1))
y_scaled = minMaxScaler.fit_transform(y_transformed.reshape(-1, 1)).flatten()


# =====================================================================
# #3. Diagnostic Reporting & Plotting
# =====================================================================

plotting.plot_performance_gain(
    initial_y_raw.max(), 
    new_y, 
    historical_std=initial_y_raw.std(), 
    use_zscore=False, 
    question=str(q)
)

reports = metrics.process_and_filter_top_m(initial_x_raw, initial_y_raw, new_x, new_y, M=5)
print("Pipeline Top M Performance Reports:\n", reports)

plotting.plot_xy(x_raw, y_scaled)

pca(q, x_raw, y_raw)

# =====================================================================
# #4. Surrogate Modeling Implementations With KFOLD Validation
# =====================================================================
k_fold_validation = []
num_folds = x_raw.shape[0]
# Test SVR with C=10 (Fixed parameter to match comment)
fold_train_losses, fold_val_losses = model.run_cross_validation(
    x_raw, y_scaled, model_type="svr", num_folds=num_folds, C=10.0, epsilon=0.01
)
k_fold_validation.append({
    "model": "SVR", 
    "parameter": "C:10, epsilon:0.01", 
    "train_loss": fold_train_losses, 
    "val_loss": fold_val_losses
})

# Test XGBoost with shallow trees to prevent overfitting your small dataset
fold_train_losses, fold_val_losses = model.run_cross_validation(
    x_raw, y_scaled, model_type="xgboost", num_folds=num_folds, max_depth=3, n_estimators=70
)
k_fold_validation.append({
    "model": "XGBoost", 
    "parameter": "max_depth:2, n_estimators:50", 
    "train_loss": fold_train_losses, 
    "val_loss": fold_val_losses
})

# Test PyTorch ensemble
fold_train_losses, fold_val_losses = model.run_cross_validation(
    x_raw, y_scaled, model_type="ensemble", num_folds=num_folds, 
    K=1, k_1=1, hidden_dim=4, lr=0.005, max_epochs=2000, tolerance=1e-5, activation="Tanh"
)
k_fold_validation.append({
    "model": "NN", 
    "parameter": "hidden_dim:4, lr:0.005, activation:Tanh", 
    "train_loss": fold_train_losses, 
    "val_loss": fold_val_losses
})

# Test LinearRegression (Degree 1)
fold_train_losses, fold_val_losses = model.run_cross_validation(
    x_raw, y_scaled, model_type="linear", num_folds=num_folds, degree=1
)
k_fold_validation.append({
    "model": "linear", 
    "parameter": "degree:1", 
    "train_loss": fold_train_losses, 
    "val_loss": fold_val_losses
})

# Test LinearRegression (Degree 2) - FIXED SYNTAX HERE
fold_train_losses, fold_val_losses = model.run_cross_validation(
    x_raw, y_scaled, model_type="linear", num_folds=num_folds, degree=2
)
k_fold_validation.append({
    "model": "linear", 
    "parameter": "degree:2", 
    "train_loss": fold_train_losses, 
    "val_loss": fold_val_losses
})

# Convert to Pandas DataFrame
k_fold_validation_pd = pd.DataFrame(k_fold_validation)
k_fold_validation_pd["mean_train_mse"] = k_fold_validation_pd["train_loss"].apply(np.mean)
k_fold_validation_pd["mean_val_mse"] = k_fold_validation_pd["val_loss"].apply(np.mean)
selected_columns = ["model", "parameter", "mean_val_mse"]

print(k_fold_validation_pd[selected_columns])

# =====================================================================
# #5. Surrogate Modeling Implementations
# =====================================================================

# Find the row index of the absolute minimum validation error
best_row_idx = k_fold_validation_pd["mean_val_mse"].idxmin()
best_run = k_fold_validation_pd.loc[best_row_idx]

print("=== BEST MODEL SELECTION ===")
print(f"Winning Architecture: {best_run['model']}")
print(f"Winning Parameters:   {best_run['parameter']}")
print(f"Best CV Validation MSE: {best_run['mean_val_mse']:.6f}\n")

# Extract the exact winning configuration
best_model_type = best_run["model"].lower()

# Map string display names back to your train_surrogate tokens
model_mapping = {"nn": "ensemble", "svr": "svr", "xgboost": "xgboost", "linear": "linear"}
target_model_token = model_mapping[best_model_type]

if target_model_token == "svr":
    best_kwargs = {"C": 10.0, "epsilon": 0.01} # Matches your exact winning configuration
elif target_model_token == "xgboost":
    best_kwargs = {"max_depth": 2, "n_estimators": 50}
elif target_model_token == "ensemble":
    best_kwargs = {"K": 1, "k_1": 1, "hidden_dim": 4, "lr": 0.005, "max_epochs": 2000, "tolerance": 1e-5, "activation": "Tanh"}
elif target_model_token == "linear":
    # Check if it was degree 1 or degree 2 from the parameter text string
    deg = 2 if "degree:2" in best_run["parameter"] else 1
    best_kwargs = {"degree": deg}

print(f"=== Retraining Final Model on 100% of Dataset... ===")
# Train on ALL the data (x_raw, y_scaled) using the optimized hyperparameters
final_production_model = model.train_surrogate(x_raw, y_scaled, model_type=target_model_token, **best_kwargs)

# Train on a GP for referrence
if normalize_data:
    kernel = RBF(length_scale=0.05, length_scale_bounds=(0.001, 1))
else:
    # Open up the bounds to allow the GP to see the wider physical scale
    kernel = RBF(length_scale=1.0, length_scale_bounds=(0.1, 10.0))
    
gp_ref = model.train_surrogate(x_raw, y_scaled, model_type="gp", kernel=kernel, noise=noise_assumption)   

# =====================================================================
# #6. Full-Grid Latent Predictions
# =====================================================================
print(f"=== Generating Predictions Across the X-Grid ===")
# Predict on your full xgrid evaluation space
if target_model_token == "ensemble":
    with torch.no_grad():
        y_grid_pred_scaled, _ = predict_ensemble_pt(final_production_model, x_grid)
        y_pred_scaled, _ = predict_ensemble_pt(final_production_model, x_raw)
        
else:
    y_grid_pred_scaled = final_production_model.predict(x_grid).flatten()
    y_pred_scaled = final_production_model.predict(x_raw).flatten()

residual_error = y_scaled - y_pred_scaled
gp = model.train_surrogate(x_raw, residual_error, model_type = "gp")
_, y_grid_sigma_scaled = gp.predict(x_grid, return_std=True)

gp_mean, gp_sigma = gp_ref.predict(x_grid, return_std = True)

# =====================================================================
# #7. Acquisition Function Search Logic
# =====================================================================
ei_scores = metrics.expected_improvement(x_grid, y_max=y_scaled.max(), default_mean=y_grid_pred_scaled, default_std=y_grid_sigma_scaled)
ucb =  metrics.ucb(y_grid_pred_scaled, y_grid_sigma_scaled, 2)
ucb_gp =  metrics.ucb(gp_mean, gp_sigma, 1)

# =====================================================================
# #8. Output Registry Determinations
# =====================================================================
# Extract optimal coordinate suggestions across different surrogates
x = np.round(x_grid[np.argmax(ei_scores)], 6)
x_mean_only = np.round(x_grid[np.argmax(y_grid_pred_scaled)], 6)
x_ucb = np.round(x_grid[np.argmax(ucb)], 6)
x_ucb_gp = np.round(x_grid[np.argmax(ucb_gp)], 6)
# Format explicit multi-coordinate display outputs
print("\n=== OPTIMIZATION ACQUISITION HIGHLIGHTS ===")
print(f"Q{q} EI Coordinate: ", "-".join([str(item) for item in x]))
print(f"Q{q} UCB Coordinate: ", "-".join([str(item) for item in x_ucb]))
print(f"Q{q} Mean Only Coordinate: ", "-".join([str(item) for item in x_mean_only]))
print(f"Q{q} GP UCB Coordinate: ", "-".join([str(item) for item in x_ucb_gp]))
print("Historical Maximum Training Sample -> Coordinate:", x_raw[np.argmax(y_scaled)])



FileNotFoundError: One or both of the input data paths do not exist.

In [8]:
# =====================================================================
# #1. Loading Data
# =====================================================================
q = 2
noise_assumption = 1e-6  # Ensure globally tracked parameters are defined upfront
normalize_data = True

# Load initial baseline files explicitly
initial_x_raw, initial_y_raw = dp.load_xy(
    f"../data/initial_data/function_{q}/initial_inputs.npy", 
    f"../data/initial_data/function_{q}/initial_outputs.npy"
)

# Extract newly recorded registry submissions for this question
new_x = np.array(input_registry[f"Q{q}"])
new_y = np.array(output_registry[f"Q{q}"])

# Combine baseline and new batches into a single active matrix
x_raw = np.vstack([initial_x_raw, new_x])
y_raw = np.append(initial_y_raw, new_y)

# FIXED: Generate grid *after* x_raw has been initialized to avoid NameError
x_grid = dp.generate_x_grid(x_raw.shape[1], m=1000)

# Track pipeline iteration metrics
all_y.append({
    "max": initial_y_raw.max(), 
    "st": initial_y_raw.std(), 
    "mean": y_raw.mean(), 
    "X_shape": x_raw.shape
})

# =====================================================================
# #2. Data Preparation
# =====================================================================
# Robust Outlier elimination using your new Median Absolute Deviation filter
x_raw, y_raw = dp.remove_outliers(x_raw, y_raw, method="mad", THRESHOLD=3)
plotting.plot_xy(x_raw, y_raw)

# Transform target tracking coordinate space

y_scaled = scaler.fit_transform(y_raw.reshape(-1, 1)).flatten() if normalize_data == True else y_raw
plotting.plot_xy(x_raw, y_scaled)
pca(q, x_raw, y_raw)
# =====================================================================
# #3. Diagnostic Reporting & Plotting
# =====================================================================

plotting.plot_performance_gain(
    initial_y_raw.max(), 
    new_y, 
    historical_std=initial_y_raw.std(), 
    use_zscore=False, 
    question=str(q)
)

reports = metrics.process_and_filter_top_m(initial_x_raw, initial_y_raw, new_x, new_y, M=5)
print("Pipeline Top M Performance Reports:\n", reports)



# =====================================================================
# #4. Surrogate Modeling Implementations With KFOLD Validation
# =====================================================================
k_fold_validation = []
num_folds = x_raw.shape[0]
# Test SVR with C=10 (Fixed parameter to match comment)
fold_train_losses, fold_val_losses = model.run_cross_validation(
    x_raw, y_scaled, model_type="svr", num_folds=num_folds, C=10.0, epsilon=0.01
)
k_fold_validation.append({
    "model": "SVR", 
    "parameter": "C:10, epsilon:0.01", 
    "train_loss": fold_train_losses, 
    "val_loss": fold_val_losses
})

# Test XGBoost with shallow trees to prevent overfitting your small dataset
fold_train_losses, fold_val_losses = model.run_cross_validation(
    x_raw, y_scaled, model_type="xgboost", num_folds=num_folds, max_depth=3, n_estimators=70
)
k_fold_validation.append({
    "model": "XGBoost", 
    "parameter": "max_depth:2, n_estimators:50", 
    "train_loss": fold_train_losses, 
    "val_loss": fold_val_losses
})

# Test PyTorch ensemble
fold_train_losses, fold_val_losses = model.run_cross_validation(
    x_raw, y_scaled, model_type="ensemble", num_folds=num_folds, 
    K=1, k_1=1, hidden_dim=4, lr=0.005, max_epochs=2000, tolerance=1e-5, activation="Tanh"
)
k_fold_validation.append({
    "model": "NN", 
    "parameter": "hidden_dim:4, lr:0.005, activation:Tanh", 
    "train_loss": fold_train_losses, 
    "val_loss": fold_val_losses
})

# Test LinearRegression (Degree 1)
fold_train_losses, fold_val_losses = model.run_cross_validation(
    x_raw, y_scaled, model_type="linear", num_folds=num_folds, degree=1
)
k_fold_validation.append({
    "model": "linear", 
    "parameter": "degree:1", 
    "train_loss": fold_train_losses, 
    "val_loss": fold_val_losses
})

# Test LinearRegression (Degree 2) - FIXED SYNTAX HERE
fold_train_losses, fold_val_losses = model.run_cross_validation(
    x_raw, y_scaled, model_type="linear", num_folds=num_folds, degree=2
)
k_fold_validation.append({
    "model": "linear", 
    "parameter": "degree:2", 
    "train_loss": fold_train_losses, 
    "val_loss": fold_val_losses
})

# Convert to Pandas DataFrame
k_fold_validation_pd = pd.DataFrame(k_fold_validation)
k_fold_validation_pd["mean_train_mse"] = k_fold_validation_pd["train_loss"].apply(np.mean)
k_fold_validation_pd["mean_val_mse"] = k_fold_validation_pd["val_loss"].apply(np.mean)
selected_columns = ["model", "parameter", "mean_val_mse"]

print(k_fold_validation_pd[selected_columns])

# =====================================================================
# #5. Surrogate Modeling Implementations
# =====================================================================

# Find the row index of the absolute minimum validation error
best_row_idx = k_fold_validation_pd["mean_val_mse"].idxmin()
best_run = k_fold_validation_pd.loc[best_row_idx]

print("=== BEST MODEL SELECTION ===")
print(f"Winning Architecture: {best_run['model']}")
print(f"Winning Parameters:   {best_run['parameter']}")
print(f"Best CV Validation MSE: {best_run['mean_val_mse']:.6f}\n")

# Extract the exact winning configuration
best_model_type = best_run["model"].lower()

# Map string display names back to your train_surrogate tokens
model_mapping = {"nn": "ensemble", "svr": "svr", "xgboost": "xgboost", "linear": "linear"}
target_model_token = model_mapping[best_model_type]

if target_model_token == "svr":
    best_kwargs = {"C": 10.0, "epsilon": 0.01} # Matches your exact winning configuration
elif target_model_token == "xgboost":
    best_kwargs = {"max_depth": 2, "n_estimators": 50}
elif target_model_token == "ensemble":
    best_kwargs = {"K": 1, "k_1": 1, "hidden_dim": 4, "lr": 0.005, "max_epochs": 2000, "tolerance": 1e-5, "activation": "Tanh"}
elif target_model_token == "linear":
    # Check if it was degree 1 or degree 2 from the parameter text string
    deg = 2 if "degree:2" in best_run["parameter"] else 1
    best_kwargs = {"degree": deg}

print(f"=== Retraining Final Model on 100% of Dataset... ===")
# Train on ALL the data (x_raw, y_scaled) using the optimized hyperparameters
final_production_model = model.train_surrogate(x_raw, y_scaled, model_type=target_model_token, **best_kwargs)

# Train on a GP for referrence
if normalize_data:
    kernel = RBF(length_scale=0.05, length_scale_bounds=(0.0001, 1))
else:
    # Open up the bounds to allow the GP to see the wider physical scale
    kernel = RBF(length_scale=1.0, length_scale_bounds=(0.1, 10.0))
    
gp_ref = model.train_surrogate(x_raw, y_scaled, model_type="gp", kernel=kernel, noise=noise_assumption)   

# =====================================================================
# #6. Full-Grid Latent Predictions
# =====================================================================
print(f"=== Generating Predictions Across the X-Grid ===")
# Predict on your full xgrid evaluation space
if target_model_token == "ensemble":
    with torch.no_grad():
        y_grid_pred_scaled, _ = predict_ensemble_pt(final_production_model, x_grid)
        y_pred_scaled, _ = predict_ensemble_pt(final_production_model, x_raw)
        
else:
    y_grid_pred_scaled = final_production_model.predict(x_grid).flatten()
    y_pred_scaled = final_production_model.predict(x_raw).flatten()

residual_error = y_scaled - y_pred_scaled
gp = model.train_surrogate(x_raw, residual_error, model_type = "gp")
_, y_grid_sigma_scaled = gp.predict(x_grid, return_std=True)

gp_mean, gp_sigma = gp_ref.predict(x_grid, return_std = True)

# =====================================================================
# #7. Acquisition Function Search Logic
# =====================================================================
ei_scores = metrics.expected_improvement(x_grid, y_max=y_scaled.max(), default_mean=y_grid_pred_scaled, default_std=y_grid_sigma_scaled)
ucb =  metrics.ucb(y_grid_pred_scaled, y_grid_sigma_scaled, 2)
ucb_gp =  metrics.ucb(gp_mean, gp_sigma, 2)

# =====================================================================
# #8. Output Registry Determinations
# =====================================================================
# Extract optimal coordinate suggestions across different surrogates
x = np.round(x_grid[np.argmax(ei_scores)], 6)
x_mean_only = np.round(x_grid[np.argmax(y_grid_pred_scaled)], 6)
x_ucb = np.round(x_grid[np.argmax(ucb)], 6)
x_ucb_gp = np.round(x_grid[np.argmax(ucb_gp)], 6)
# Format explicit multi-coordinate display outputs
print("\n=== OPTIMIZATION ACQUISITION HIGHLIGHTS ===")
print(f"Q{q} EI Coordinate: ", "-".join([str(item) for item in x]))
print(f"Q{q} UCB Coordinate: ", "-".join([str(item) for item in x_ucb]))
print(f"Q{q} Mean Only Coordinate: ", "-".join([str(item) for item in x_mean_only]))
print(f"Q{q} GP UCB Coordinate: ", "-".join([str(item) for item in x_ucb_gp]))
print("Historical Maximum Training Sample -> Coordinate:", x_raw[np.argmax(y_scaled)])

FileNotFoundError: One or both of the input data paths do not exist.

In [ ]:
# =====================================================================
# #1. Loading Data
# =====================================================================
q = 3
noise_assumption = 1e-6  # Ensure globally tracked parameters are defined upfront
normalize_data = True
minMaxScaler = MinMaxScaler(feature_range=(0, 1))
# Load initial baseline files explicitly
initial_x_raw, initial_y_raw = dp.load_xy(
    f"../data/initial_data/function_{q}/initial_inputs.npy", 
    f"../data/initial_data/function_{q}/initial_outputs.npy"
)

# Extract newly recorded registry submissions for this question
new_x = np.array(input_registry[f"Q{q}"])
new_y = np.array(output_registry[f"Q{q}"])

# Combine baseline and new batches into a single active matrix
x_raw = np.vstack([initial_x_raw, new_x])
y_raw = np.append(initial_y_raw, new_y)

# FIXED: Generate grid *after* x_raw has been initialized to avoid NameError
x_grid = dp.generate_x_grid(x_raw.shape[1], m=100)

# Track pipeline iteration metrics
all_y.append({
    "max": initial_y_raw.max(), 
    "st": initial_y_raw.std(), 
    "mean": y_raw.mean(), 
    "X_shape": x_raw.shape
})

# =====================================================================
# #2. Data Preparation
# =====================================================================
# Robust Outlier elimination using your new Median Absolute Deviation filter
#x_raw, y_raw = dp.remove_outliers(x_raw, y_raw, method="mad", THRESHOLD=3)
plotting.plot_xy(x_raw, y_raw)

# Transform target tracking coordinate space
#y_scaled = scaler.fit_transform(y_raw.reshape(-1, 1)).flatten()
y_scaled = minMaxScaler.fit_transform(y_raw.reshape(-1, 1)).flatten() if normalize_data == True else y_raw
plotting.plot_xy(x_raw, y_scaled)
# =====================================================================
# #3. Diagnostic Reporting & Plotting
# =====================================================================

plotting.plot_performance_gain(
    initial_y_raw.max(), 
    new_y, 
    historical_std=initial_y_raw.std(), 
    use_zscore=False, 
    question=str(q)
)

reports = metrics.process_and_filter_top_m(initial_x_raw, initial_y_raw, new_x, new_y, M=5)
print("Pipeline Top M Performance Reports:\n", reports)

pca(q, x_raw, y_raw)

# =====================================================================
# #4. Surrogate Modeling Implementations With KFOLD Validation
# =====================================================================
k_fold_validation = []
num_folds = 10
# Test SVR with C=10 (Fixed parameter to match comment)
fold_train_losses, fold_val_losses = model.run_cross_validation(
    x_raw, y_scaled, model_type="svr", num_folds=num_folds, C=10.0, epsilon=0.01, kernel="linear"
)
k_fold_validation.append({
    "model": "SVR", 
    "parameter": "C:10, epsilon:0.01", 
    "train_loss": fold_train_losses, 
    "val_loss": fold_val_losses
})

# Test XGBoost with shallow trees to prevent overfitting your small dataset
fold_train_losses, fold_val_losses = model.run_cross_validation(
    x_raw, y_scaled, model_type="xgboost", num_folds=num_folds, max_depth=3, n_estimators=70
)
k_fold_validation.append({
    "model": "XGBoost", 
    "parameter": "max_depth:2, n_estimators:50", 
    "train_loss": fold_train_losses, 
    "val_loss": fold_val_losses
})

# Test PyTorch ensemble
fold_train_losses, fold_val_losses = model.run_cross_validation(
    x_raw, y_scaled, model_type="ensemble", num_folds=num_folds, 
    K=1, k_1=1, hidden_dim=4, lr=0.005, max_epochs=2000, tolerance=1e-5, activation="Tanh"
)
k_fold_validation.append({
    "model": "NN", 
    "parameter": "hidden_dim:4, lr:0.005, activation:Tanh", 
    "train_loss": fold_train_losses, 
    "val_loss": fold_val_losses
})

# Test LinearRegression (Degree 1)
fold_train_losses, fold_val_losses = model.run_cross_validation(
    x_raw, y_scaled, model_type="linear", num_folds=num_folds, degree=1
)
k_fold_validation.append({
    "model": "linear", 
    "parameter": "degree:1", 
    "train_loss": fold_train_losses, 
    "val_loss": fold_val_losses
})

# Test LinearRegression (Degree 2) - FIXED SYNTAX HERE
fold_train_losses, fold_val_losses = model.run_cross_validation(
    x_raw, y_scaled, model_type="linear", num_folds=num_folds, degree=2
)
k_fold_validation.append({
    "model": "linear", 
    "parameter": "degree:2", 
    "train_loss": fold_train_losses, 
    "val_loss": fold_val_losses
})

# Convert to Pandas DataFrame
k_fold_validation_pd = pd.DataFrame(k_fold_validation)
k_fold_validation_pd["mean_train_mse"] = k_fold_validation_pd["train_loss"].apply(np.mean)
k_fold_validation_pd["mean_val_mse"] = k_fold_validation_pd["val_loss"].apply(np.mean)
selected_columns = ["model", "parameter", "mean_val_mse"]

print(k_fold_validation_pd[selected_columns])

# =====================================================================
# #5. Surrogate Modeling Implementations
# =====================================================================

# Find the row index of the absolute minimum validation error
best_row_idx = k_fold_validation_pd["mean_val_mse"].idxmin()
best_run = k_fold_validation_pd.loc[best_row_idx]

print("=== BEST MODEL SELECTION ===")
print(f"Winning Architecture: {best_run['model']}")
print(f"Winning Parameters:   {best_run['parameter']}")
print(f"Best CV Validation MSE: {best_run['mean_val_mse']:.6f}\n")

# Extract the exact winning configuration
best_model_type = best_run["model"].lower()

# Map string display names back to your train_surrogate tokens
model_mapping = {"nn": "ensemble", "svr": "svr", "xgboost": "xgboost", "linear": "linear"}
target_model_token = model_mapping[best_model_type]

if target_model_token == "svr":
    best_kwargs = {"C": 10.0, "epsilon": 0.01, kernel:"linear"} # Matches your exact winning configuration
elif target_model_token == "xgboost":
    best_kwargs = {"max_depth": 2, "n_estimators": 50}
elif target_model_token == "ensemble":
    best_kwargs = {"K": 1, "k_1": 1, "hidden_dim": 4, "lr": 0.005, "max_epochs": 2000, "tolerance": 1e-5, "activation": "Tanh"}
elif target_model_token == "linear":
    # Check if it was degree 1 or degree 2 from the parameter text string
    deg = 2 if "degree:2" in best_run["parameter"] else 1
    best_kwargs = {"degree": deg}

print(f"=== Retraining Final Model on 100% of Dataset... ===")
# Train on ALL the data (x_raw, y_scaled) using the optimized hyperparameters
final_production_model = model.train_surrogate(x_raw, y_scaled, model_type=target_model_token, **best_kwargs)

# Train on a GP for referrence
if normalize_data:
    kernel = RBF(length_scale=0.05, length_scale_bounds=(0.001, 1))
else:
    # Open up the bounds to allow the GP to see the wider physical scale
    kernel = RBF(length_scale=1.0, length_scale_bounds=(0.1, 10.0))
    
gp_ref = model.train_surrogate(x_raw, y_scaled, model_type="gp", kernel=kernel, noise=noise_assumption)   

# =====================================================================
# #6. Full-Grid Latent Predictions
# =====================================================================
print(f"=== Generating Predictions Across the X-Grid ===")
# Predict on your full xgrid evaluation space
if target_model_token == "ensemble":
    with torch.no_grad():
        y_grid_pred_scaled, _ = model.predict_ensemble_pt(final_production_model, x_grid)
        y_pred_scaled, _ = model.predict_ensemble_pt(final_production_model, x_raw)
        
else:
    y_grid_pred_scaled = final_production_model.predict(x_grid).flatten()
    y_pred_scaled = final_production_model.predict(x_raw).flatten()

residual_error = y_scaled - y_pred_scaled
gp = model.train_surrogate(x_raw, residual_error, model_type = "gp")
_, y_grid_sigma_scaled = gp.predict(x_grid, return_std=True)

gp_mean, gp_sigma = gp_ref.predict(x_grid, return_std = True)

# =====================================================================
# #7. Acquisition Function Search Logic
# =====================================================================
ei_scores = metrics.expected_improvement(x_grid, y_max=y_scaled.max(), default_mean=y_grid_pred_scaled, default_std=y_grid_sigma_scaled)
ucb =  metrics.ucb(y_grid_pred_scaled, y_grid_sigma_scaled, 2)
ucb_gp =  metrics.ucb(gp_mean, gp_sigma, 1.2)

# =====================================================================
# #8. Output Registry Determinations
# =====================================================================
# Extract optimal coordinate suggestions across different surrogates
x = np.round(x_grid[np.argmax(ei_scores)], 6)
x_mean_only = np.round(x_grid[np.argmax(y_grid_pred_scaled)], 6)
x_ucb = np.round(x_grid[np.argmax(ucb)], 6)
x_ucb_gp = np.round(x_grid[np.argmax(ucb_gp)], 6)
# Format explicit multi-coordinate display outputs
print("\n=== OPTIMIZATION ACQUISITION HIGHLIGHTS ===")
print(f"Q{q} EI Coordinate: ", "-".join([str(item) for item in x]))
print(f"Q{q} UCB Coordinate: ", "-".join([str(item) for item in x_ucb]))
print(f"Q{q} Mean Only Coordinate: ", "-".join([str(item) for item in x_mean_only]))
print(f"Q{q} GP UCB Coordinate: ", "-".join([str(item) for item in x_ucb_gp]))
print("Historical Maximum Training Sample -> Coordinate:", x_raw[np.argmax(y_scaled)])

In [ ]:
# =====================================================================
# #1. Loading Data
# =====================================================================
q = 4
noise_assumption = 1e-6  # Ensure globally tracked parameters are defined upfront
normalize_data = True

# Load initial baseline files explicitly
initial_x_raw, initial_y_raw = dp.load_xy(
    f"../data/initial_data/function_{q}/initial_inputs.npy", 
    f"../data/initial_data/function_{q}/initial_outputs.npy"
)

# Extract newly recorded registry submissions for this question
new_x = np.array(input_registry[f"Q{q}"])
new_y = np.array(output_registry[f"Q{q}"])

# Combine baseline and new batches into a single active matrix
x_raw = np.vstack([initial_x_raw, new_x])
y_raw = np.append(initial_y_raw, new_y)

# FIXED: Generate grid *after* x_raw has been initialized to avoid NameError
x_grid = dp.generate_x_grid(x_raw.shape[1], m=40)

# Track pipeline iteration metrics
all_y.append({
    "max": initial_y_raw.max(), 
    "st": initial_y_raw.std(), 
    "mean": y_raw.mean(), 
    "X_shape": x_raw.shape
})

# =====================================================================
# #2. Data Preparation
# =====================================================================
# Robust Outlier elimination using your new Median Absolute Deviation filter
x_raw, y_raw = dp.remove_outliers(x_raw, y_raw, method="mad", THRESHOLD=3)
plotting.plot_xy(x_raw, y_raw)

# Transform target tracking coordinate space
#y_scaled = scaler.fit_transform(y_raw.reshape(-1, 1)).flatten()
y_scaled = scaler.fit_transform(y_raw.reshape(-1, 1)).flatten() if normalize_data == True else y_raw
plotting.plot_xy(x_raw, y_scaled)
pca(q, x_raw, y_raw)
# =====================================================================
# #3. Diagnostic Reporting & Plotting
# =====================================================================

plotting.plot_performance_gain(
    initial_y_raw.max(), 
    new_y, 
    historical_std=initial_y_raw.std(), 
    use_zscore=False, 
    question=str(q)
)

reports = metrics.process_and_filter_top_m(initial_x_raw, initial_y_raw, new_x, new_y, M=5)
print("Pipeline Top M Performance Reports:\n", reports)



# =====================================================================
# #4. Surrogate Modeling Implementations With KFOLD Validation
# =====================================================================
k_fold_validation = []
num_folds = 5
# Test SVR with C=10 (Fixed parameter to match comment)
fold_train_losses, fold_val_losses = model.run_cross_validation(
    x_raw, y_scaled, model_type="svr", num_folds=num_folds, C=10.0, epsilon=0.01
)
k_fold_validation.append({
    "model": "SVR", 
    "parameter": "C:10, epsilon:0.01", 
    "train_loss": fold_train_losses, 
    "val_loss": fold_val_losses
})

# Test XGBoost with shallow trees to prevent overfitting your small dataset
fold_train_losses, fold_val_losses = model.run_cross_validation(
    x_raw, y_scaled, model_type="xgboost", num_folds=num_folds, max_depth=3, n_estimators=70
)
k_fold_validation.append({
    "model": "XGBoost", 
    "parameter": "max_depth:2, n_estimators:50", 
    "train_loss": fold_train_losses, 
    "val_loss": fold_val_losses
})

# Test PyTorch ensemble
fold_train_losses, fold_val_losses = model.run_cross_validation(
    x_raw, y_scaled, model_type="ensemble", num_folds=num_folds, 
    K=1, k_1=1, hidden_dim=4, lr=0.005, max_epochs=2000, tolerance=1e-5, activation="Tanh"
)
k_fold_validation.append({
    "model": "NN", 
    "parameter": "hidden_dim:4, lr:0.005, activation:Tanh", 
    "train_loss": fold_train_losses, 
    "val_loss": fold_val_losses
})

# Test LinearRegression (Degree 1)
fold_train_losses, fold_val_losses = model.run_cross_validation(
    x_raw, y_scaled, model_type="linear", num_folds=num_folds, degree=1
)
k_fold_validation.append({
    "model": "linear", 
    "parameter": "degree:1", 
    "train_loss": fold_train_losses, 
    "val_loss": fold_val_losses
})

# Test LinearRegression (Degree 2) - FIXED SYNTAX HERE
fold_train_losses, fold_val_losses = model.run_cross_validation(
    x_raw, y_scaled, model_type="linear", num_folds=num_folds, degree=2
)
k_fold_validation.append({
    "model": "linear", 
    "parameter": "degree:2", 
    "train_loss": fold_train_losses, 
    "val_loss": fold_val_losses
})

# Convert to Pandas DataFrame
k_fold_validation_pd = pd.DataFrame(k_fold_validation)
k_fold_validation_pd["mean_train_mse"] = k_fold_validation_pd["train_loss"].apply(np.mean)
k_fold_validation_pd["mean_val_mse"] = k_fold_validation_pd["val_loss"].apply(np.mean)
selected_columns = ["model", "parameter", "mean_val_mse"]

print(k_fold_validation_pd[selected_columns])

# =====================================================================
# #5. Surrogate Modeling Implementations
# =====================================================================

# Find the row index of the absolute minimum validation error
best_row_idx = k_fold_validation_pd["mean_val_mse"].idxmin()
best_run = k_fold_validation_pd.loc[best_row_idx]

print("=== BEST MODEL SELECTION ===")
print(f"Winning Architecture: {best_run['model']}")
print(f"Winning Parameters:   {best_run['parameter']}")
print(f"Best CV Validation MSE: {best_run['mean_val_mse']:.6f}\n")

# Extract the exact winning configuration
best_model_type = best_run["model"].lower()

# Map string display names back to your train_surrogate tokens
model_mapping = {"nn": "ensemble", "svr": "svr", "xgboost": "xgboost", "linear": "linear"}
target_model_token = model_mapping[best_model_type]

if target_model_token == "svr":
    best_kwargs = {"C": 10.0, "epsilon": 0.01} # Matches your exact winning configuration
elif target_model_token == "xgboost":
    best_kwargs = {"max_depth": 2, "n_estimators": 50}
elif target_model_token == "ensemble":
    best_kwargs = {"K": 1, "k_1": 1, "hidden_dim": 4, "lr": 0.005, "max_epochs": 2000, "tolerance": 1e-5, "activation": "Tanh"}
elif target_model_token == "linear":
    # Check if it was degree 1 or degree 2 from the parameter text string
    deg = 2 if "degree:2" in best_run["parameter"] else 1
    best_kwargs = {"degree": deg}

print(f"=== Retraining Final Model on 100% of Dataset... ===")
# Train on ALL the data (x_raw, y_scaled) using the optimized hyperparameters
final_production_model = model.train_surrogate(x_raw, y_scaled, model_type=target_model_token, **best_kwargs)

# Train on a GP for referrence
if normalize_data:
    kernel = RBF(length_scale=0.05, length_scale_bounds=(0.001, 1))
else:
    # Open up the bounds to allow the GP to see the wider physical scale
    kernel = RBF(length_scale=1.0, length_scale_bounds=(0.1, 10.0))
    
gp_ref = model.train_surrogate(x_raw, y_scaled, model_type="gp", kernel=kernel, noise=noise_assumption)   

# =====================================================================
# #6. Full-Grid Latent Predictions
# =====================================================================
print(f"=== Generating Predictions Across the X-Grid ===")
# Predict on your full xgrid evaluation space
if target_model_token == "ensemble":
    with torch.no_grad():
        y_grid_pred_scaled, _ = predict_ensemble_pt(final_production_model, x_grid)
        y_pred_scaled, _ = predict_ensemble_pt(final_production_model, x_raw)
        
else:
    y_grid_pred_scaled = final_production_model.predict(x_grid).flatten()
    y_pred_scaled = final_production_model.predict(x_raw).flatten()

residual_error = y_scaled - y_pred_scaled
gp = model.train_surrogate(x_raw, residual_error, model_type = "gp")
_, y_grid_sigma_scaled = gp.predict(x_grid, return_std=True)

gp_mean, gp_sigma = gp_ref.predict(x_grid, return_std = True)

# =====================================================================
# #7. Acquisition Function Search Logic
# =====================================================================
ei_scores = metrics.expected_improvement(x_grid, y_max=y_scaled.max(), default_mean=y_grid_pred_scaled, default_std=y_grid_sigma_scaled)
ucb =  metrics.ucb(y_grid_pred_scaled, y_grid_sigma_scaled, 2)
ucb_gp =  metrics.ucb(gp_mean, gp_sigma, 2)

# =====================================================================
# #8. Output Registry Determinations
# =====================================================================
# Extract optimal coordinate suggestions across different surrogates
x = np.round(x_grid[np.argmax(ei_scores)], 6)
x_mean_only = np.round(x_grid[np.argmax(y_grid_pred_scaled)], 6)
x_ucb = np.round(x_grid[np.argmax(ucb)], 6)
x_ucb_gp = np.round(x_grid[np.argmax(ucb_gp)], 6)
# Format explicit multi-coordinate display outputs
print("\n=== OPTIMIZATION ACQUISITION HIGHLIGHTS ===")
print(f"Q{q} EI Coordinate: ", "-".join([str(item) for item in x]))
print(f"Q{q} UCB Coordinate: ", "-".join([str(item) for item in x_ucb]))
print(f"Q{q} Mean Only Coordinate: ", "-".join([str(item) for item in x_mean_only]))
print(f"Q{q} GP UCB Coordinate: ", "-".join([str(item) for item in x_ucb_gp]))
print("Historical Maximum Training Sample -> Coordinate:", x_raw[np.argmax(y_scaled)])

In [ ]:
# =====================================================================
# #1. Loading Data
# =====================================================================
q = 5
noise_assumption = 1e-6  # Ensure globally tracked parameters are defined upfront
normalize_data = True

# Load initial baseline files explicitly
initial_x_raw, initial_y_raw = dp.load_xy(
    f"../data/initial_data/function_{q}/initial_inputs.npy", 
    f"../data/initial_data/function_{q}/initial_outputs.npy"
)

# Extract newly recorded registry submissions for this question
new_x = np.array(input_registry[f"Q{q}"])
new_y = np.array(output_registry[f"Q{q}"])

# Combine baseline and new batches into a single active matrix
x_raw = np.vstack([initial_x_raw, new_x])
y_raw = np.append(initial_y_raw, new_y)

# FIXED: Generate grid *after* x_raw has been initialized to avoid NameError
x_grid = dp.generate_x_grid(x_raw.shape[1], m=30)

# Track pipeline iteration metrics
all_y.append({
    "max": initial_y_raw.max(), 
    "st": initial_y_raw.std(), 
    "mean": y_raw.mean(), 
    "X_shape": x_raw.shape
})

# =====================================================================
# #2. Data Preparation
# =====================================================================
# Robust Outlier elimination using your new Median Absolute Deviation filter
x_raw, y_raw = dp.remove_outliers(x_raw, y_raw, method="mad", THRESHOLD=3)
plotting.plot_xy(x_raw, y_raw)

# Transform target tracking coordinate space
#y_scaled = scaler.fit_transform(y_raw.reshape(-1, 1)).flatten()
y_scaled = scaler.fit_transform(y_raw.reshape(-1, 1)).flatten() if normalize_data == True else y_raw
plotting.plot_xy(x_raw, y_scaled)
pca(q, x_raw, y_raw)
# =====================================================================
# #3. Diagnostic Reporting & Plotting
# =====================================================================

plotting.plot_performance_gain(
    initial_y_raw.max(), 
    new_y, 
    historical_std=initial_y_raw.std(), 
    use_zscore=False, 
    question=str(q)
)

reports = metrics.process_and_filter_top_m(initial_x_raw, initial_y_raw, new_x, new_y, M=5)
print("Pipeline Top M Performance Reports:\n", reports)



# =====================================================================
# #4. Surrogate Modeling Implementations With KFOLD Validation
# =====================================================================
k_fold_validation = []
num_folds = 5
# Test SVR with C=10 (Fixed parameter to match comment)
fold_train_losses, fold_val_losses = model.run_cross_validation(
    x_raw, y_scaled, model_type="svr", num_folds=num_folds, C=10.0, epsilon=0.01
)
k_fold_validation.append({
    "model": "SVR", 
    "parameter": "C:10, epsilon:0.01", 
    "train_loss": fold_train_losses, 
    "val_loss": fold_val_losses
})

# Test XGBoost with shallow trees to prevent overfitting your small dataset
fold_train_losses, fold_val_losses = model.run_cross_validation(
    x_raw, y_scaled, model_type="xgboost", num_folds=num_folds, max_depth=3, n_estimators=70
)
k_fold_validation.append({
    "model": "XGBoost", 
    "parameter": "max_depth:2, n_estimators:50", 
    "train_loss": fold_train_losses, 
    "val_loss": fold_val_losses
})

# Test PyTorch ensemble
fold_train_losses, fold_val_losses = model.run_cross_validation(
    x_raw, y_scaled, model_type="ensemble", num_folds=num_folds, 
    K=1, k_1=1, hidden_dim=4, lr=0.005, max_epochs=2000, tolerance=1e-5, activation="Tanh"
)
k_fold_validation.append({
    "model": "NN", 
    "parameter": "hidden_dim:4, lr:0.005, activation:Tanh", 
    "train_loss": fold_train_losses, 
    "val_loss": fold_val_losses
})

# Test LinearRegression (Degree 1)
fold_train_losses, fold_val_losses = model.run_cross_validation(
    x_raw, y_scaled, model_type="linear", num_folds=num_folds, degree=1
)
k_fold_validation.append({
    "model": "linear", 
    "parameter": "degree:1", 
    "train_loss": fold_train_losses, 
    "val_loss": fold_val_losses
})

# Test LinearRegression (Degree 2) - FIXED SYNTAX HERE
fold_train_losses, fold_val_losses = model.run_cross_validation(
    x_raw, y_scaled, model_type="linear", num_folds=num_folds, degree=2
)
k_fold_validation.append({
    "model": "linear", 
    "parameter": "degree:2", 
    "train_loss": fold_train_losses, 
    "val_loss": fold_val_losses
})

# Convert to Pandas DataFrame
k_fold_validation_pd = pd.DataFrame(k_fold_validation)
k_fold_validation_pd["mean_train_mse"] = k_fold_validation_pd["train_loss"].apply(np.mean)
k_fold_validation_pd["mean_val_mse"] = k_fold_validation_pd["val_loss"].apply(np.mean)
selected_columns = ["model", "parameter", "mean_val_mse"]

print(k_fold_validation_pd[selected_columns])

# =====================================================================
# #5. Surrogate Modeling Implementations
# =====================================================================

# Find the row index of the absolute minimum validation error
best_row_idx = k_fold_validation_pd["mean_val_mse"].idxmin()
best_run = k_fold_validation_pd.loc[best_row_idx]

print("=== BEST MODEL SELECTION ===")
print(f"Winning Architecture: {best_run['model']}")
print(f"Winning Parameters:   {best_run['parameter']}")
print(f"Best CV Validation MSE: {best_run['mean_val_mse']:.6f}\n")

# Extract the exact winning configuration
best_model_type = best_run["model"].lower()

# Map string display names back to your train_surrogate tokens
model_mapping = {"nn": "ensemble", "svr": "svr", "xgboost": "xgboost", "linear": "linear"}
target_model_token = model_mapping[best_model_type]

if target_model_token == "svr":
    best_kwargs = {"C": 10.0, "epsilon": 0.01} # Matches your exact winning configuration
elif target_model_token == "xgboost":
    best_kwargs = {"max_depth": 2, "n_estimators": 50}
elif target_model_token == "ensemble":
    best_kwargs = {"K": 1, "k_1": 1, "hidden_dim": 4, "lr": 0.005, "max_epochs": 2000, "tolerance": 1e-5, "activation": "Tanh"}
elif target_model_token == "linear":
    # Check if it was degree 1 or degree 2 from the parameter text string
    deg = 2 if "degree:2" in best_run["parameter"] else 1
    best_kwargs = {"degree": deg}

print(f"=== Retraining Final Model on 100% of Dataset... ===")
# Train on ALL the data (x_raw, y_scaled) using the optimized hyperparameters
final_production_model = model.train_surrogate(x_raw, y_scaled, model_type=target_model_token, **best_kwargs)

# Train on a GP for referrence
if normalize_data:
    kernel = RBF(length_scale=0.05, length_scale_bounds=(0.001, 1))
else:
    # Open up the bounds to allow the GP to see the wider physical scale
    kernel = RBF(length_scale=1.0, length_scale_bounds=(0.1, 10.0))
    
gp_ref = model.train_surrogate(x_raw, y_scaled, model_type="gp", kernel=kernel, noise=noise_assumption)   

# =====================================================================
# #6. Full-Grid Latent Predictions
# =====================================================================
print(f"=== Generating Predictions Across the X-Grid ===")
# Predict on your full xgrid evaluation space
if target_model_token == "ensemble":
    with torch.no_grad():
        y_grid_pred_scaled, _ = model.predict_ensemble_pt(final_production_model, x_grid)
        y_pred_scaled, _ = model.predict_ensemble_pt(final_production_model, x_raw)
        
else:
    y_grid_pred_scaled = final_production_model.predict(x_grid).flatten()
    y_pred_scaled = final_production_model.predict(x_raw).flatten()

residual_error = y_scaled - y_pred_scaled
gp = model.train_surrogate(x_raw, residual_error, model_type = "gp")
_, y_grid_sigma_scaled = gp.predict(x_grid, return_std=True)

gp_mean, gp_sigma = gp_ref.predict(x_grid, return_std = True)

# =====================================================================
# #7. Acquisition Function Search Logic
# =====================================================================
ei_scores = metrics.expected_improvement(x_grid, y_max=y_scaled.max(), default_mean=y_grid_pred_scaled, default_std=y_grid_sigma_scaled)
ucb =  metrics.ucb(y_grid_pred_scaled, y_grid_sigma_scaled, 1)
ucb_gp =  metrics.ucb(gp_mean, gp_sigma, 1)

# =====================================================================
# #8. Output Registry Determinations
# =====================================================================
# Extract optimal coordinate suggestions across different surrogates
x = np.round(x_grid[np.argmax(ei_scores)], 6)
x_mean_only = np.round(x_grid[np.argmax(y_grid_pred_scaled)], 6)
x_ucb = np.round(x_grid[np.argmax(ucb)], 6)
x_ucb_gp = np.round(x_grid[np.argmax(ucb_gp)], 6)
# Format explicit multi-coordinate display outputs
print("\n=== OPTIMIZATION ACQUISITION HIGHLIGHTS ===")
print(f"Q{q} EI Coordinate: ", "-".join([str(item) for item in x]))
print(f"Q{q} UCB Coordinate: ", "-".join([str(item) for item in x_ucb]))
print(f"Q{q} Mean Only Coordinate: ", "-".join([str(item) for item in x_mean_only]))
print(f"Q{q} GP UCB Coordinate: ", "-".join([str(item) for item in x_ucb_gp]))
print("Historical Maximum Training Sample -> Coordinate:", x_raw[np.argmax(y_scaled)])

In [ ]:
# =====================================================================
# #1. Loading Data
# =====================================================================
q = 6
noise_assumption = 1e-6  # Ensure globally tracked parameters are defined upfront
normalize_data = True

# Load initial baseline files explicitly
initial_x_raw, initial_y_raw = dp.load_xy(
    f"../data/initial_data/function_{q}/initial_inputs.npy", 
    f"../data/initial_data/function_{q}/initial_outputs.npy"
)

# Extract newly recorded registry submissions for this question
new_x = np.array(input_registry[f"Q{q}"])
new_y = np.array(output_registry[f"Q{q}"])

# Combine baseline and new batches into a single active matrix
x_raw = np.vstack([initial_x_raw, new_x])
y_raw = np.append(initial_y_raw, new_y)

# FIXED: Generate grid *after* x_raw has been initialized to avoid NameError
x_grid = dp.generate_x_grid(x_raw.shape[1], m=23)

# Track pipeline iteration metrics
all_y.append({
    "max": initial_y_raw.max(), 
    "st": initial_y_raw.std(), 
    "mean": y_raw.mean(), 
    "X_shape": x_raw.shape
})

# =====================================================================
# #2. Data Preparation
# =====================================================================
# Robust Outlier elimination using your new Median Absolute Deviation filter
x_raw, y_raw = dp.remove_outliers(x_raw, y_raw, method="mad", THRESHOLD=3)
plotting.plot_xy(x_raw, y_raw)

# Transform target tracking coordinate space
#y_scaled = scaler.fit_transform(y_raw.reshape(-1, 1)).flatten()
y_scaled = scaler.fit_transform(y_raw.reshape(-1, 1)).flatten() if normalize_data == True else y_raw
plotting.plot_xy(x_raw, y_scaled)
pca(q, x_raw, y_raw)
# =====================================================================
# #3. Diagnostic Reporting & Plotting
# =====================================================================

plotting.plot_performance_gain(
    initial_y_raw.max(), 
    new_y, 
    historical_std=initial_y_raw.std(), 
    use_zscore=False, 
    question=str(q)
)

reports = metrics.process_and_filter_top_m(initial_x_raw, initial_y_raw, new_x, new_y, M=5)
print("Pipeline Top M Performance Reports:\n", reports)



# =====================================================================
# #4. Surrogate Modeling Implementations With KFOLD Validation
# =====================================================================
k_fold_validation = []
num_folds = 5
# Test SVR with C=10 (Fixed parameter to match comment)
fold_train_losses, fold_val_losses = model.run_cross_validation(
    x_raw, y_scaled, model_type="svr", num_folds=num_folds, C=10.0, epsilon=0.01
)
k_fold_validation.append({
    "model": "SVR", 
    "parameter": "C:10, epsilon:0.01", 
    "train_loss": fold_train_losses, 
    "val_loss": fold_val_losses
})

# Test XGBoost with shallow trees to prevent overfitting your small dataset
fold_train_losses, fold_val_losses = model.run_cross_validation(
    x_raw, y_scaled, model_type="xgboost", num_folds=num_folds, max_depth=3, n_estimators=70
)
k_fold_validation.append({
    "model": "XGBoost", 
    "parameter": "max_depth:2, n_estimators:50", 
    "train_loss": fold_train_losses, 
    "val_loss": fold_val_losses
})

# Test PyTorch ensemble
fold_train_losses, fold_val_losses = model.run_cross_validation(
    x_raw, y_scaled, model_type="ensemble", num_folds=num_folds, 
    K=1, k_1=1, hidden_dim=4, lr=0.005, max_epochs=2000, tolerance=1e-5, activation="Tanh"
)
k_fold_validation.append({
    "model": "NN", 
    "parameter": "hidden_dim:4, lr:0.005, activation:Tanh", 
    "train_loss": fold_train_losses, 
    "val_loss": fold_val_losses
})

# Test LinearRegression (Degree 1)
fold_train_losses, fold_val_losses = model.run_cross_validation(
    x_raw, y_scaled, model_type="linear", num_folds=num_folds, degree=1
)
k_fold_validation.append({
    "model": "linear", 
    "parameter": "degree:1", 
    "train_loss": fold_train_losses, 
    "val_loss": fold_val_losses
})

# Test LinearRegression (Degree 2) - FIXED SYNTAX HERE
fold_train_losses, fold_val_losses = model.run_cross_validation(
    x_raw, y_scaled, model_type="linear", num_folds=num_folds, degree=2
)
k_fold_validation.append({
    "model": "linear", 
    "parameter": "degree:2", 
    "train_loss": fold_train_losses, 
    "val_loss": fold_val_losses
})

# Convert to Pandas DataFrame
k_fold_validation_pd = pd.DataFrame(k_fold_validation)
k_fold_validation_pd["mean_train_mse"] = k_fold_validation_pd["train_loss"].apply(np.mean)
k_fold_validation_pd["mean_val_mse"] = k_fold_validation_pd["val_loss"].apply(np.mean)
selected_columns = ["model", "parameter", "mean_val_mse"]

print(k_fold_validation_pd[selected_columns])

# =====================================================================
# #5. Surrogate Modeling Implementations
# =====================================================================

# Find the row index of the absolute minimum validation error
best_row_idx = k_fold_validation_pd["mean_val_mse"].idxmin()
best_run = k_fold_validation_pd.loc[best_row_idx]

print("=== BEST MODEL SELECTION ===")
print(f"Winning Architecture: {best_run['model']}")
print(f"Winning Parameters:   {best_run['parameter']}")
print(f"Best CV Validation MSE: {best_run['mean_val_mse']:.6f}\n")

# Extract the exact winning configuration
best_model_type = best_run["model"].lower()

# Map string display names back to your train_surrogate tokens
model_mapping = {"nn": "ensemble", "svr": "svr", "xgboost": "xgboost", "linear": "linear"}
target_model_token = model_mapping[best_model_type]

if target_model_token == "svr":
    best_kwargs = {"C": 10.0, "epsilon": 0.01} # Matches your exact winning configuration
elif target_model_token == "xgboost":
    best_kwargs = {"max_depth": 2, "n_estimators": 50}
elif target_model_token == "ensemble":
    best_kwargs = {"K": 1, "k_1": 1, "hidden_dim": 4, "lr": 0.005, "max_epochs": 2000, "tolerance": 1e-5, "activation": "Tanh"}
elif target_model_token == "linear":
    # Check if it was degree 1 or degree 2 from the parameter text string
    deg = 2 if "degree:2" in best_run["parameter"] else 1
    best_kwargs = {"degree": deg}

print(f"=== Retraining Final Model on 100% of Dataset... ===")
# Train on ALL the data (x_raw, y_scaled) using the optimized hyperparameters
final_production_model = model.train_surrogate(x_raw, y_scaled, model_type=target_model_token, **best_kwargs)

# Train on a GP for referrence
if normalize_data:
    kernel = RBF(length_scale=0.05, length_scale_bounds=(0.001, 1))
else:
    # Open up the bounds to allow the GP to see the wider physical scale
    kernel = RBF(length_scale=1.0, length_scale_bounds=(0.1, 10.0))
    
gp_ref = model.train_surrogate(x_raw, y_scaled, model_type="gp", kernel=kernel, noise=noise_assumption)   

# =====================================================================
# #6. Full-Grid Latent Predictions
# =====================================================================
print(f"=== Generating Predictions Across the X-Grid ===")
# Predict on your full xgrid evaluation space
if target_model_token == "ensemble":
    with torch.no_grad():
        y_grid_pred_scaled, _ = predict_ensemble_pt(final_production_model, x_grid)
        y_pred_scaled, _ = predict_ensemble_pt(final_production_model, x_raw)
        
else:
    y_grid_pred_scaled = final_production_model.predict(x_grid).flatten()
    y_pred_scaled = final_production_model.predict(x_raw).flatten()

residual_error = y_scaled - y_pred_scaled
gp = model.train_surrogate(x_raw, residual_error, model_type = "gp")
_, y_grid_sigma_scaled = gp.predict(x_grid, return_std=True)

gp_mean, gp_sigma = gp_ref.predict(x_grid, return_std = True)

# =====================================================================
# #7. Acquisition Function Search Logic
# =====================================================================
ei_scores = metrics.expected_improvement(x_grid, y_max=y_scaled.max(), default_mean=y_grid_pred_scaled, default_std=y_grid_sigma_scaled)
ucb =  metrics.ucb(y_grid_pred_scaled, y_grid_sigma_scaled, 2)
ucb_gp =  metrics.ucb(gp_mean, gp_sigma, 2)

# =====================================================================
# #8. Output Registry Determinations
# =====================================================================
# Extract optimal coordinate suggestions across different surrogates
x = np.round(x_grid[np.argmax(ei_scores)], 6)
x_mean_only = np.round(x_grid[np.argmax(y_grid_pred_scaled)], 6)
x_ucb = np.round(x_grid[np.argmax(ucb)], 6)
x_ucb_gp = np.round(x_grid[np.argmax(ucb_gp)], 6)
# Format explicit multi-coordinate display outputs
print("\n=== OPTIMIZATION ACQUISITION HIGHLIGHTS ===")
print(f"Q{q} EI Coordinate: ", "-".join([str(item) for item in x]))
print(f"Q{q} UCB Coordinate: ", "-".join([str(item) for item in x_ucb]))
print(f"Q{q} Mean Only Coordinate: ", "-".join([str(item) for item in x_mean_only]))
print(f"Q{q} GP UCB Coordinate: ", "-".join([str(item) for item in x_ucb_gp]))
print("Historical Maximum Training Sample -> Coordinate:", x_raw[np.argmax(y_scaled)])

In [ ]:
# =====================================================================
# #1. Loading Data
# =====================================================================
q = 7
noise_assumption = 1e-6  # Ensure globally tracked parameters are defined upfront
normalize_data = True

# Load initial baseline files explicitly
initial_x_raw, initial_y_raw = dp.load_xy(
    f"../data/initial_data/function_{q}/initial_inputs.npy", 
    f"../data/initial_data/function_{q}/initial_outputs.npy"
)
# Extract newly recorded registry submissions for this question
new_x = np.array(input_registry[f"Q{q}"])
new_y = np.array(output_registry[f"Q{q}"])

# Combine baseline and new batches into a single active matrix
x_raw = np.vstack([initial_x_raw, new_x])
y_raw = np.append(initial_y_raw, new_y)

x_grid = dp.generate_x_grid(x_raw.shape[1], m=13)
X_local_eval_grid = np.random.normal(loc= x_raw[np.argmax(y_raw)], scale=0.1, size=(400000, x_raw.shape[1]))
X_local_eval_grid = np.clip(X_local_eval_grid, 0.0, 1.0)

# Track pipeline iteration metrics
all_y.append({
    "max": initial_y_raw.max(), 
    "st": initial_y_raw.std(), 
    "mean": y_raw.mean(), 
    "X_shape": x_raw.shape
})

# =====================================================================
# #2. Data Preparation
# =====================================================================
# Robust Outlier elimination using your new Median Absolute Deviation filter
x_raw, y_raw = dp.remove_outliers(x_raw, y_raw, method="mad", THRESHOLD=3)
plotting.plot_xy(x_raw, y_raw)

# Transform target tracking coordinate space
#y_scaled = scaler.fit_transform(y_raw.reshape(-1, 1)).flatten()
y_scaled = scaler.fit_transform(y_raw.reshape(-1, 1)).flatten() if normalize_data == True else y_raw
plotting.plot_xy(x_raw, y_scaled)
pca(q, x_raw, y_raw)
# =====================================================================
# #3. Diagnostic Reporting & Plotting
# =====================================================================

plotting.plot_performance_gain(
    initial_y_raw.max(), 
    new_y, 
    historical_std=initial_y_raw.std(), 
    use_zscore=False, 
    question=str(q)
)

reports = metrics.process_and_filter_top_m(initial_x_raw, initial_y_raw, new_x, new_y, M=5)
print("Pipeline Top M Performance Reports:\n", reports)



# =====================================================================
# #4. Surrogate Modeling Implementations With KFOLD Validation
# =====================================================================
k_fold_validation = []
num_folds = 5
# Test SVR with C=10 (Fixed parameter to match comment)
fold_train_losses, fold_val_losses = model.run_cross_validation(
    x_raw, y_scaled, model_type="svr", num_folds=num_folds, C=10.0, epsilon=0.01
)
k_fold_validation.append({
    "model": "SVR", 
    "parameter": "C:10, epsilon:0.01", 
    "train_loss": fold_train_losses, 
    "val_loss": fold_val_losses
})

# Test XGBoost with shallow trees to prevent overfitting your small dataset
fold_train_losses, fold_val_losses = model.run_cross_validation(
    x_raw, y_scaled, model_type="xgboost", num_folds=num_folds, max_depth=3, n_estimators=70
)
k_fold_validation.append({
    "model": "XGBoost", 
    "parameter": "max_depth:2, n_estimators:50", 
    "train_loss": fold_train_losses, 
    "val_loss": fold_val_losses
})

# Test PyTorch ensemble
fold_train_losses, fold_val_losses = model.run_cross_validation(
    x_raw, y_scaled, model_type="ensemble", num_folds=num_folds, 
    K=1, k_1=1, hidden_dim=4, lr=0.005, max_epochs=2000, tolerance=1e-5, activation="Tanh"
)
k_fold_validation.append({
    "model": "NN", 
    "parameter": "hidden_dim:4, lr:0.005, activation:Tanh", 
    "train_loss": fold_train_losses, 
    "val_loss": fold_val_losses
})

# Test LinearRegression (Degree 1)
fold_train_losses, fold_val_losses = model.run_cross_validation(
    x_raw, y_scaled, model_type="linear", num_folds=num_folds, degree=1
)
k_fold_validation.append({
    "model": "linear", 
    "parameter": "degree:1", 
    "train_loss": fold_train_losses, 
    "val_loss": fold_val_losses
})

# Test LinearRegression (Degree 2) - FIXED SYNTAX HERE
fold_train_losses, fold_val_losses = model.run_cross_validation(
    x_raw, y_scaled, model_type="linear", num_folds=num_folds, degree=2
)
k_fold_validation.append({
    "model": "linear", 
    "parameter": "degree:2", 
    "train_loss": fold_train_losses, 
    "val_loss": fold_val_losses
})

# Convert to Pandas DataFrame
k_fold_validation_pd = pd.DataFrame(k_fold_validation)
k_fold_validation_pd["mean_train_mse"] = k_fold_validation_pd["train_loss"].apply(np.mean)
k_fold_validation_pd["mean_val_mse"] = k_fold_validation_pd["val_loss"].apply(np.mean)
selected_columns = ["model", "parameter", "mean_val_mse"]

print(k_fold_validation_pd[selected_columns])

# =====================================================================
# #5. Surrogate Modeling Implementations
# =====================================================================

# Find the row index of the absolute minimum validation error
best_row_idx = k_fold_validation_pd["mean_val_mse"].idxmin()
best_run = k_fold_validation_pd.loc[best_row_idx]

print("=== BEST MODEL SELECTION ===")
print(f"Winning Architecture: {best_run['model']}")
print(f"Winning Parameters:   {best_run['parameter']}")
print(f"Best CV Validation MSE: {best_run['mean_val_mse']:.6f}\n")

# Extract the exact winning configuration
best_model_type = best_run["model"].lower()

# Map string display names back to your train_surrogate tokens
model_mapping = {"nn": "ensemble", "svr": "svr", "xgboost": "xgboost", "linear": "linear"}
target_model_token = model_mapping[best_model_type]

if target_model_token == "svr":
    best_kwargs = {"C": 10.0, "epsilon": 0.01} # Matches your exact winning configuration
elif target_model_token == "xgboost":
    best_kwargs = {"max_depth": 2, "n_estimators": 50}
elif target_model_token == "ensemble":
    best_kwargs = {"K": 1, "k_1": 1, "hidden_dim": 4, "lr": 0.005, "max_epochs": 2000, "tolerance": 1e-5, "activation": "Tanh"}
elif target_model_token == "linear":
    # Check if it was degree 1 or degree 2 from the parameter text string
    deg = 2 if "degree:2" in best_run["parameter"] else 1
    best_kwargs = {"degree": deg}

print(f"=== Retraining Final Model on 100% of Dataset... ===")
# Train on ALL the data (x_raw, y_scaled) using the optimized hyperparameters
final_production_model = model.train_surrogate(x_raw, y_scaled, model_type=target_model_token, **best_kwargs)

# Train on a GP for referrence
if normalize_data:
    kernel = RBF(length_scale=0.05, length_scale_bounds=(0.001, 1))
else:
    # Open up the bounds to allow the GP to see the wider physical scale
    kernel = RBF(length_scale=1.0, length_scale_bounds=(0.1, 10.0))
    
gp_ref = model.train_surrogate(x_raw, y_scaled, model_type="gp", kernel=kernel, noise=noise_assumption)   

# =====================================================================
# #6. Full-Grid Latent Predictions
# =====================================================================
print(f"=== Generating Predictions Across the X-Grid ===")
# Predict on your full xgrid evaluation space
if target_model_token == "ensemble":
    with torch.no_grad():
        y_grid_pred_scaled, _ = predict_ensemble_pt(final_production_model, X_local_eval_grid)
        y_pred_scaled, _ = predict_ensemble_pt(final_production_model, x_raw)
        
else:
    y_grid_pred_scaled = final_production_model.predict(X_local_eval_grid).flatten()
    y_pred_scaled = final_production_model.predict(x_raw).flatten()

residual_error = y_scaled - y_pred_scaled
gp = model.train_surrogate(x_raw, residual_error, model_type = "gp")
_, y_grid_sigma_scaled = gp.predict(X_local_eval_grid, return_std=True)

gp_mean, gp_sigma = gp_ref.predict(X_local_eval_grid, return_std = True)

# =====================================================================
# #7. Acquisition Function Search Logic
# =====================================================================
ei_scores = metrics.expected_improvement(X_local_eval_grid, y_max=y_scaled.max(), default_mean=y_grid_pred_scaled, default_std=y_grid_sigma_scaled)
ucb =  metrics.ucb(y_grid_pred_scaled, y_grid_sigma_scaled, 2)
ucb_gp =  metrics.ucb(gp_mean, gp_sigma, 2)

# =====================================================================
# #8. Output Registry Determinations
# =====================================================================
# Extract optimal coordinate suggestions across different surrogates
x = np.round(X_local_eval_grid[np.argmax(ei_scores)], 6)
x_mean_only = np.round(X_local_eval_grid[np.argmax(y_grid_pred_scaled)], 6)
x_ucb = np.round(X_local_eval_grid[np.argmax(ucb)], 6)
x_ucb_gp = np.round(X_local_eval_grid[np.argmax(ucb_gp)], 6)
# Format explicit multi-coordinate display outputs
print("\n=== OPTIMIZATION ACQUISITION HIGHLIGHTS ===")
print(f"Q{q} EI Coordinate: ", "-".join([str(item) for item in x]))
print(f"Q{q} UCB Coordinate: ", "-".join([str(item) for item in x_ucb]))
print(f"Q{q} Mean Only Coordinate: ", "-".join([str(item) for item in x_mean_only]))
print(f"Q{q} GP UCB Coordinate: ", "-".join([str(item) for item in x_ucb_gp]))
print("Historical Maximum Training Sample -> Coordinate:", x_raw[np.argmax(y_scaled)])

In [ ]:
# =====================================================================
# #1. Loading Data
# =====================================================================
q = 8
noise_assumption = 1e-6  # Ensure globally tracked parameters are defined upfront
normalize_data = True

# Load initial baseline files explicitly
initial_x_raw, initial_y_raw = dp.load_xy(
    f"../data/initial_data/function_{q}/initial_inputs.npy", 
    f"../data/initial_data/function_{q}/initial_outputs.npy"
)
# Extract newly recorded registry submissions for this question
new_x = np.array(input_registry[f"Q{q}"])
new_y = np.array(output_registry[f"Q{q}"])

# Combine baseline and new batches into a single active matrix
x_raw = np.vstack([initial_x_raw, new_x])
y_raw = np.append(initial_y_raw, new_y)

# FIXED: Generate grid *after* x_raw has been initialized to avoid NameError
#x_grid = dp.generate_x_grid(x_raw.shape[1], m=13)
X_local_eval_grid = np.random.normal(loc= x_raw[np.argmax(y_raw)], scale=0.1, size=(400000, x_raw.shape[1]))
X_local_eval_grid = np.clip(X_local_eval_grid, 0.0, 1.0)

# Track pipeline iteration metrics
all_y.append({
    "max": initial_y_raw.max(), 
    "st": initial_y_raw.std(), 
    "mean": y_raw.mean(), 
    "X_shape": x_raw.shape
})

# =====================================================================
# #2. Data Preparation
# =====================================================================
# Robust Outlier elimination using your new Median Absolute Deviation filter
x_raw, y_raw = dp.remove_outliers(x_raw, y_raw, method="mad", THRESHOLD=3)
plotting.plot_xy(x_raw, y_raw)

# Transform target tracking coordinate space
#y_scaled = scaler.fit_transform(y_raw.reshape(-1, 1)).flatten()
y_scaled = scaler.fit_transform(y_raw.reshape(-1, 1)).flatten() if normalize_data == True else y_raw
plotting.plot_xy(x_raw, y_scaled)
pca(q, x_raw, y_raw)
# =====================================================================
# #3. Diagnostic Reporting & Plotting
# =====================================================================

plotting.plot_performance_gain(
    initial_y_raw.max(), 
    new_y, 
    historical_std=initial_y_raw.std(), 
    use_zscore=False, 
    question=str(q)
)

reports = metrics.process_and_filter_top_m(initial_x_raw, initial_y_raw, new_x, new_y, M=5)
print("Pipeline Top M Performance Reports:\n", reports)



# =====================================================================
# #4. Surrogate Modeling Implementations With KFOLD Validation
# =====================================================================
k_fold_validation = []
num_folds = 5
# Test SVR with C=10 (Fixed parameter to match comment)
fold_train_losses, fold_val_losses = model.run_cross_validation(
    x_raw, y_scaled, model_type="svr", num_folds=num_folds, C=10.0, epsilon=0.01
)
k_fold_validation.append({
    "model": "SVR", 
    "parameter": "C:10, epsilon:0.01", 
    "train_loss": fold_train_losses, 
    "val_loss": fold_val_losses
})

# Test XGBoost with shallow trees to prevent overfitting your small dataset
fold_train_losses, fold_val_losses = model.run_cross_validation(
    x_raw, y_scaled, model_type="xgboost", num_folds=num_folds, max_depth=3, n_estimators=70
)
k_fold_validation.append({
    "model": "XGBoost", 
    "parameter": "max_depth:2, n_estimators:50", 
    "train_loss": fold_train_losses, 
    "val_loss": fold_val_losses
})

# Test PyTorch ensemble
fold_train_losses, fold_val_losses = model.run_cross_validation(
    x_raw, y_scaled, model_type="ensemble", num_folds=num_folds, 
    K=1, k_1=1, hidden_dim=4, lr=0.005, max_epochs=2000, tolerance=1e-5, activation="Tanh"
)
k_fold_validation.append({
    "model": "NN", 
    "parameter": "hidden_dim:4, lr:0.005, activation:Tanh", 
    "train_loss": fold_train_losses, 
    "val_loss": fold_val_losses
})

# Test LinearRegression (Degree 1)
fold_train_losses, fold_val_losses = model.run_cross_validation(
    x_raw, y_scaled, model_type="linear", num_folds=num_folds, degree=1
)
k_fold_validation.append({
    "model": "linear", 
    "parameter": "degree:1", 
    "train_loss": fold_train_losses, 
    "val_loss": fold_val_losses
})

# Test LinearRegression (Degree 2) - FIXED SYNTAX HERE
fold_train_losses, fold_val_losses = model.run_cross_validation(
    x_raw, y_scaled, model_type="linear", num_folds=num_folds, degree=2
)
k_fold_validation.append({
    "model": "linear", 
    "parameter": "degree:2", 
    "train_loss": fold_train_losses, 
    "val_loss": fold_val_losses
})

# Convert to Pandas DataFrame
k_fold_validation_pd = pd.DataFrame(k_fold_validation)
k_fold_validation_pd["mean_train_mse"] = k_fold_validation_pd["train_loss"].apply(np.mean)
k_fold_validation_pd["mean_val_mse"] = k_fold_validation_pd["val_loss"].apply(np.mean)
selected_columns = ["model", "parameter", "mean_val_mse"]

print(k_fold_validation_pd[selected_columns])

# =====================================================================
# #5. Surrogate Modeling Implementations
# =====================================================================

# Find the row index of the absolute minimum validation error
best_row_idx = k_fold_validation_pd["mean_val_mse"].idxmin()
best_run = k_fold_validation_pd.loc[best_row_idx]

print("=== BEST MODEL SELECTION ===")
print(f"Winning Architecture: {best_run['model']}")
print(f"Winning Parameters:   {best_run['parameter']}")
print(f"Best CV Validation MSE: {best_run['mean_val_mse']:.6f}\n")

# Extract the exact winning configuration
best_model_type = best_run["model"].lower()

# Map string display names back to your train_surrogate tokens
model_mapping = {"nn": "ensemble", "svr": "svr", "xgboost": "xgboost", "linear": "linear"}
target_model_token = model_mapping[best_model_type]

if target_model_token == "svr":
    best_kwargs = {"C": 10.0, "epsilon": 0.01} # Matches your exact winning configuration
elif target_model_token == "xgboost":
    best_kwargs = {"max_depth": 2, "n_estimators": 50}
elif target_model_token == "ensemble":
    best_kwargs = {"K": 1, "k_1": 1, "hidden_dim": 4, "lr": 0.005, "max_epochs": 2000, "tolerance": 1e-5, "activation": "Tanh"}
elif target_model_token == "linear":
    # Check if it was degree 1 or degree 2 from the parameter text string
    deg = 2 if "degree:2" in best_run["parameter"] else 1
    best_kwargs = {"degree": deg}

print(f"=== Retraining Final Model on 100% of Dataset... ===")
# Train on ALL the data (x_raw, y_scaled) using the optimized hyperparameters
final_production_model = model.train_surrogate(x_raw, y_scaled, model_type=target_model_token, **best_kwargs)

# Train on a GP for referrence
if normalize_data:
    kernel = RBF(length_scale=0.05, length_scale_bounds=(0.001, 5))
else:
    # Open up the bounds to allow the GP to see the wider physical scale
    kernel = RBF(length_scale=1.0, length_scale_bounds=(0.1, 10.0))
    
gp_ref = model.train_surrogate(x_raw, y_scaled, model_type="gp", kernel=kernel, noise=noise_assumption)   

# =====================================================================
# #6. Full-Grid Latent Predictions
# =====================================================================
print(f"=== Generating Predictions Across the X-Grid ===")
# Predict on your full xgrid evaluation space
if target_model_token == "ensemble":
    with torch.no_grad():
        y_grid_pred_scaled, _ = predict_ensemble_pt(final_production_model, X_local_eval_grid)
        y_pred_scaled, _ = predict_ensemble_pt(final_production_model, x_raw)
        
else:
    y_grid_pred_scaled = final_production_model.predict(X_local_eval_grid).flatten()
    y_pred_scaled = final_production_model.predict(x_raw).flatten()

residual_error = y_scaled - y_pred_scaled
gp = model.train_surrogate(x_raw, residual_error, model_type = "gp")
_, y_grid_sigma_scaled = gp.predict(X_local_eval_grid, return_std=True)

gp_mean, gp_sigma = gp_ref.predict(X_local_eval_grid, return_std = True)

# =====================================================================
# #7. Acquisition Function Search Logic
# =====================================================================
ei_scores = metrics.expected_improvement(X_local_eval_grid, y_max=y_scaled.max(), default_mean=y_grid_pred_scaled, default_std=y_grid_sigma_scaled)
ucb =  metrics.ucb(y_grid_pred_scaled, y_grid_sigma_scaled, 2)
ucb_gp =  metrics.ucb(gp_mean, gp_sigma, 2)

# =====================================================================
# #8. Output Registry Determinations
# =====================================================================
# Extract optimal coordinate suggestions across different surrogates
x = np.round(X_local_eval_grid[np.argmax(ei_scores)], 6)
x_mean_only = np.round(X_local_eval_grid[np.argmax(y_grid_pred_scaled)], 6)
x_ucb = np.round(X_local_eval_grid[np.argmax(ucb)], 6)
x_ucb_gp = np.round(X_local_eval_grid[np.argmax(ucb_gp)], 6)
# Format explicit multi-coordinate display outputs
print("\n=== OPTIMIZATION ACQUISITION HIGHLIGHTS ===")
print(f"Q{q} EI Coordinate: ", "-".join([str(item) for item in x]))
print(f"Q{q} UCB Coordinate: ", "-".join([str(item) for item in x_ucb]))
print(f"Q{q} Mean Only Coordinate: ", "-".join([str(item) for item in x_mean_only]))
print(f"Q{q} GP UCB Coordinate: ", "-".join([str(item) for item in x_ucb_gp]))
print("Historical Maximum Training Sample -> Coordinate:", x_raw[np.argmax(y_scaled)])

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from scipy.optimize import minimize
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(x_raw)
model = LinearRegression().fit(X_poly, y_raw)
feature_names = poly.get_feature_names_out(input_features=[f"x{i}" for i in range(8)])
coefs = model.coef_

n_dims = x_raw.shape[1]  # 8
H = np.zeros((n_dims, n_dims))
b = np.zeros(n_dims)
c = float(model.intercept_)
coefs = model.coef_

# 2. poly.powers_ gives the exact degree for each term without string parsing
# Shape of poly.powers_: (n_polynomial_features, 8)
for power_row, coef in zip(poly.powers_, coefs):
    degree = np.sum(power_row)
    active_indices = np.where(power_row > 0)[0]
    
    if degree == 1:
        # Linear term: x_i
        i = active_indices[0]
        b[i] = coef
        
    elif degree == 2:
        if len(active_indices) == 1:
            # Pure quadratic term: x_i^2 -> coef * x_i^2 = (1/2) * H_ii * x_i^2
            i = active_indices[0]
            H[i, i] = 2.0 * coef
        else:
            # Interaction term: x_i * x_j -> coef * x_i * x_j = H_ij * x_i * x_j
            i, j = active_indices
            H[i, j] = coef
            H[j, i] = coef

# 3. Check if matrix H is negative definite (true unconstrained peak)
eigenvalues = np.linalg.eigvalsh(H)  # eigvalsh is faster and numerically stable for symmetric matrices
is_strictly_concave = np.all(eigenvalues < 0)

print("Hessian Eigenvalues:", eigenvalues)
print("Has unique unconstrained global maximum:", is_strictly_concave)

bounds = list(zip(x_raw.min(axis=0), x_raw.max(axis=0)))

# Objective function to maximize (minimize negative y)
def neg_poly(x):
    # - (0.5 * x^T H x + b^T x + c)
    return -(0.5 * np.dot(x, np.dot(H, x)) + np.dot(b, x) + c)

def neg_poly_grad(x):
    # Analytical gradient: -(H x + b)
    return -(np.dot(H, x) + b)

# Initial guess (e.g., center of bounds or unconstrained critical point)
if is_strictly_concave:
    x0 = -np.linalg.solve(H, b)
    x0 = np.clip(x0, [bnd[0] for bnd in bounds], [bnd[1] for bnd in bounds])
else:
    x0 = np.mean(bounds, axis=1)

# Solve exactly using Sequential Least Squares Programming (SLSQP)
res = minimize(
    fun=neg_poly,
    x0=x0,
    jac=neg_poly_grad,
    bounds=bounds,
    method="SLSQP",
    options={"ftol": 1e-12, "disp": False},
)

exact_x_max = res.x
exact_y_max = -res.fun

print("Exact Maximum 8D Coordinate:", exact_x_max)
print("Exact Maximum Value:", exact_y_max)


In [ ]:
from PIL import Image
import os

# Put your 8 image paths here
image_paths = [f"Percentage Improvement Scale - Q{i+1}.png" for i in range(8)]

# Open the first image to get dimensions (assuming all are identical)
img = Image.open(image_paths[0])
img_w, img_h = img.size

# Create a blank canvas for a 3x3 grid (with a white background)
grid_w = img_w * 3
grid_h = img_h * 3
grid_image = Image.new('RGB', (grid_w, grid_h), color='white')

# Paste the 8 images into the grid
for index, path in enumerate(image_paths):
    img = Image.open(path)

    # Calculate X and Y coordinates on the grid
    x = (index % 3) * img_w
    y = (index // 3) * img_h

    grid_image.paste(img, (x, y))

# Save the final single image
grid_image.save('final_3x3_grid.jpg')
print("Grid created successfully!")